# Phase 1: historical Joint Frozen CPU baseline

Select **Runtime > Change runtime type > CPU**. This notebook calls the pinned repository's preparation, tests, and baseline runner; it contains no model implementation. It does not run Joint TAR or ReFuTab.

Replace `PROJECT_SHA` below with the full 40-character commit of the reviewed, pushed ReFuTab source. Run the cells in order. The placeholder deliberately fails before mounting Drive or downloading anything. Mount your private Google Drive interactively; no token belongs in this notebook. Outputs are stored under `MyDrive/ReFuTab/runs/`.

A notebook file is an execution aid, not evidence that a Colab run succeeded. Review the run manifest and metrics after execution. See [the workflow](https://github.com/diyarh26/residual-guided-text-tabular/blob/main/docs/COLAB.md) for interruption and retry limitations.

In [ ]:
import re
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

PROJECT_SHA = "REPLACE_WITH_FULL_40_CHARACTER_PROJECT_SHA"
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_SHA) is None:
    raise ValueError("Set PROJECT_SHA to the full lowercase 40-character reviewed and pushed ReFuTab commit.")

REPOSITORY = "https://github.com/diyarh26/residual-guided-text-tabular.git"
PROJECT_DIR = Path("/content/refutab")
RUNS_ROOT = Path("/content/drive/MyDrive/ReFuTab/runs")
RUN_ID = "phase1-colab-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
print("Project commit:", PROJECT_SHA)
print("Run ID:", RUN_ID)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_LOG = RUNS_ROOT / "_bootstrap" / (RUN_ID + ".log")
BOOTSTRAP_LOG.parent.mkdir(parents=True, exist_ok=True)
with BOOTSTRAP_LOG.open("x", encoding="utf-8") as stream:
    stream.write(f"project_sha={PROJECT_SHA}\nrun_id={RUN_ID}\nrequested_hardware=cpu\n")
print("Persistent console log:", BOOTSTRAP_LOG)

In [ ]:
import shlex
import subprocess
import sys

def run(command, *, cwd=None):
    # Stream each source command to both the cell and persistent Drive log.
    lines = []
    with BOOTSTRAP_LOG.open("a", encoding="utf-8") as stream:
        heading = "$ " + shlex.join([str(part) for part in command])
        print(heading)
        stream.write(heading + "\n")
        stream.flush()
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                stream.write(line)
                stream.flush()
                lines.append(line)
            return_code = process.wait()
        stream.write(f"exit_code={return_code}\n")
        stream.flush()
    if return_code:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {BOOTSTRAP_LOG}")
    return "".join(lines)

if PROJECT_DIR.exists():
    raise FileExistsError("/content/refutab already exists. Use a fresh runtime or explicitly move that checkout before retrying.")
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv==0.11.24"])
run(["git", "clone", REPOSITORY, str(PROJECT_DIR)])
run(["git", "checkout", "--detach", PROJECT_SHA], cwd=PROJECT_DIR)
actual_sha = run(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR).strip()
if actual_sha != PROJECT_SHA:
    raise RuntimeError(f"Source commit mismatch: {actual_sha}")
if run(["git", "status", "--porcelain"], cwd=PROJECT_DIR).strip():
    raise RuntimeError("Source checkout must be clean before setup.")
run(["uv", "sync", "--locked", "--python", "3.11"], cwd=PROJECT_DIR)

Preparation fetches the exact historical upstream source and pinned pilot data/E5 snapshot through repository code. No upstream source is copied into the notebook. The dependency lock installs CPU Torch; a GPU runtime does not change this baseline's device. Preparation and tests must finish successfully before fitting.

In [ ]:
run(["uv", "run", "--frozen", "python", "scripts/prepare_phase1.py"], cwd=PROJECT_DIR)
run(["uv", "run", "--frozen", "python", "-m", "unittest", "discover", "-s", "tests"], cwd=PROJECT_DIR)
if run(["git", "status", "--porcelain"], cwd=PROJECT_DIR).strip():
    raise RuntimeError("Preparation or tests modified tracked source or created unignored files; inspect before fitting.")

In [ ]:
run(["uv", "run", "--frozen", "python", "scripts/run_baseline.py",
     "--config", "configs/phase1_frozen.json", "--run-id", RUN_ID,
     "--output-root", str(RUNS_ROOT)], cwd=PROJECT_DIR)
print("Review the runner's manifest, metrics, and artifacts under:", RUNS_ROOT / RUN_ID)
print("Preparation, test, and run console log:", BOOTSTRAP_LOG)

## After execution

Check the saved run status, source/upstream/data/model revisions, actual split IDs, finite metrics, environment/hardware, and artifact paths. A command returning successfully alone is insufficient evidence for the project report. Record this Colab execution and its actual cost/runtime in the experiment log. Export only reviewed lightweight summaries to GitHub; keep data, embeddings, fitted models, and credentials out of Git.

**Interruption:** Drive retains the flushed console log and any runner artifacts already saved. LightGBM's current `fit` call cannot resume from a partial checkpoint. Treat an incomplete attempt as incomplete, retain its folder, and retry the entire baseline with a new run ID. This notebook has no optimizer/checkpoint resume API. Do not claim this notebook was executed in Colab until it actually was.